# Writing Effective Agent Instructions

*Notebook 05*

Instructions are the primary way you control agent behavior.  

A well-written instruction set makes an agent reliable and predictable — a vague one produces inconsistent results that are hard to debug.

**Topics:**
- Writing durable system instructions

- Telling agents when to ask clarifying questions

- Telling agents when NOT to use a tool

- Response style constraints

- Completion criteria

- Refusal and escalation patterns

---

## 🔧 Setup

In [1]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

✅ Ready!


---

## 📋 Part 1: Durable Instructions & Completion Criteria

Instructions run on every turn. They need to hold up across a wide range of inputs, and they need to tell the agent when it's done — what format to use, how long to be, and when to stop.

### Vague vs Durable

#### Run Vague Agent

In [ ]:
message = """Summarize this for me:
AI agents combine a language model with tools, memory, and instructions to accomplish tasks.
Unlike a plain chatbot, an agent can call functions, look up data, run code, and chain multiple steps
together to reach a goal. Modern agent frameworks handle the orchestration — deciding which tool to
call, passing results between steps, and recovering from errors — so developers can focus on defining
the agent's behavior rather than the control flow."""

agent_vague = Agent(
    name="VagueAgent",
    instructions="Be helpful and summarize things.",
    model=MODEL
)

vague_result = await Runner.run(agent_vague, input=message)

print("📌 Vague instructions:")
print(vague_result.final_output)

#### Run Durable Agent

In [ ]:
durable_instructions = (
    "You are a technical writing assistant.\n"
    "When asked to summarize, produce exactly one sentence.\n"
    "Use plain language — no jargon unless the user introduced it.\n"
    "Do not add commentary or follow-up questions."
)

agent_durable = Agent(
    name="DurableAgent",
    instructions=durable_instructions,
    model=MODEL
)

durable_result = await Runner.run(agent_durable, input=message)

print("📌 Durable instructions:")
print(durable_result.final_output)
print()
print("The same before/after improvement applies to the multi-tool assistant you built in Lesson 04 — replacing its vague instructions with these durable rules makes its tool choices far more predictable.")

### 💡 Why This Works

The durable instructions specify three things that the vague version left open:

- **Format** — `produce exactly one sentence`
- **Length** — `use plain language`
- **Stop condition** — `Do not add commentary or follow-up questions`

These are *completion criteria* — they tell the agent when it's done. A durable instruction usually answers four things explicitly: role, task, output format, and what not to do.

---

Note: this agent has no real scheduling tool — it responds *as if* it can schedule, but it's only generating text. We're focused on the conversational logic here.

#### Run Agent with Incomplete Request

#### Run Agent with Incomplete Request

In [5]:
clarify_instructions = (
    "You are a scheduling assistant.\n"
    "Before booking anything, confirm: the date, time, and attendees.\n"
    "If any of these are missing, ask for them — do not assume.\n"
    "If all required details are present, confirm the booking details back to the user without asking for more information.\n"
    "When confirming, repeat the date back in the user's own words."
)

agent_clarify = Agent(
    name="SchedulingAgent",
    instructions=clarify_instructions,
    model=MODEL
)

clarify_result = await Runner.run(agent_clarify, input="Book a meeting with the team.")

print(clarify_result.final_output)

I can do that. Before I book, I need the date, the start time (and timezone), and who exactly “the team” includes (names or emails). Those three are required — I won’t assume them.

Optional: meeting duration, location or virtual link (Zoom/Teams), and an agenda or title. Do you want me to check attendees’ availability or just schedule at the time you specify?


#### Run Agent with Complete Request

### 💡 Why This Works

Same instructions, two behaviors — depending on what the user provides:

- Missing information → triggers a question
- Complete information → agent proceeds without asking

The mechanism: list the specific required fields explicitly. Without that list, "ask if unclear" produces inconsistent behavior. Use this any time acting on missing details would create the wrong calendar event, send the wrong email, or trigger any other wasted action.

---

## 🚫 Part 3: When NOT to Use a Tool

In Lesson 04 you saw that the agent decides when to call a tool. Instructions let you constrain that decision — telling the agent when to call a tool and when to answer directly.

In [ ]:
@function_tool
def lookup_product_price(product_name: str) -> str:
    """Look up the current price of a product by name."""
    prices = {
        "widget": "$9.99",
        "gadget": "$24.99",
        "doohickey": "$4.99"
    }
    return prices.get(product_name.lower(), "Product not found.")


tool_constraint_instructions = (
    "You are a product support assistant.\n"
    "Use the lookup_product_price tool only when the user asks about price.\n"
    "For general questions, answer directly without calling any tool."
)

agent_constrained = Agent(
    name="ProductAgent",
    instructions=tool_constraint_instructions,
    model=MODEL,
    tools=[lookup_product_price]
)

result_general = await Runner.run(agent_constrained, input="What is a widget?")

print("📌 General question (no tool):")
print(result_general.final_output)

result_price = await Runner.run(agent_constrained, input="How much does a widget cost?")

print("\n📌 Price question (tool called):")
print(result_price.final_output)


### 💡 Why This Works

The instruction names the condition for tool use explicitly. Without it, the agent decides on its own — and may call the tool on questions it could answer directly. On a real project this saves tokens, reduces latency, and stops the agent from calling paid or slow tools for questions it already knows how to answer. Instructions sit above tool docstrings in the decision chain — the agent first checks instructions for whether a tool is allowed, then looks at docstrings to pick which one.

## 🛑 Part 4: Refusal and Escalation Patterns

Agents need to know what they should not do, and what to do instead:

- A refusal without escalation leaves the user stuck
- An escalation without a refusal can send the agent down a path it shouldn't take

**Refusal** = the agent says "I can't do that."
**Escalation** = the agent says "I can't do that, but here's who can."

In [ ]:
refusal_instructions = (
    "You are a customer support agent for a software product.\n"
    "Only answer questions about the product — features, pricing, and troubleshooting.\n"
    "If the user asks about anything outside this scope, politely decline and suggest they contact the appropriate resource.\n"
    "If the user reports a billing dispute or account access issue, tell them to contact billing@acme.example — do not attempt to resolve it yourself."
)

agent_refusal = Agent(
    name="SupportAgent",
    instructions=refusal_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ SupportAgent ready")

#### Test: In-Scope Question

In [ ]:
result_inscope = await Runner.run(agent_refusal, input="Does your product support two-factor authentication?")

print("📌 In-scope:")
print(result_inscope.final_output)

#### Test: Out-of-Scope Question

In [ ]:
result_outofscope = await Runner.run(agent_refusal, input="Can you help me write a cover letter?")

print("📌 Out-of-scope:")
print(result_outofscope.final_output)

#### Test: Escalation Trigger

In [ ]:
result_escalate = await Runner.run(agent_refusal, input="I was charged twice last month.")

print("📌 Escalation:")
print(result_escalate.final_output)

### 💡 Why This Works

The instruction defines scope, the refusal response, and the escalation path explicitly.  

The agent doesn't have to infer any of these — which means the behavior is consistent and predictable across all three cases.

---

## 💪 Practice Exercises

### Exercise 1: Strict Formatter

*Covers: durable instructions — explicit format and completion criteria*

Write instructions for an agent that always responds in a fixed format: a one-sentence summary followed by exactly three bullet points.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Strict Formatter
# --------------------------------------------------------------
# Objective: Build an agent with instructions that enforce a fixed output format.

# TODO 1: Write instructions that require:
#            - One sentence summary
#            - Exactly three bullet points
#            - No additional commentary

# TODO 2: Create an Agent with those instructions

# TODO 3: Run it with: "Tell me about large language models"
#            and verify the output matches the format

# --- Write your code below this line ---

### Exercise 2: Scoped Support Agent

*Covers: refusal and escalation — scoping agent behavior*

Build an agent that handles cooking questions only — and escalates anything else with a specific message.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Scoped Support Agent
# --------------------------------------------------------------
# Objective: Write instructions that define scope and an escalation path.

# TODO 1: Write instructions that:
#            - Limit the agent to cooking-related questions only
#            - Define a specific refusal message for out-of-scope questions
#            - Escalate recipe disputes to: recipes@example.com

# TODO 2: Create an Agent with those instructions

# TODO 3: Test with three inputs:
#            - "How do I make pasta carbonara?"  (in-scope)
#            - "What's the weather today?"        (out-of-scope)
#            - "Your carbonara recipe is wrong." (escalation)

# --- Write your code below this line ---

## 🎯 Key Takeaways

**Durable instructions:**

- Specify format, length, and tone explicitly
- Define completion criteria so the agent knows when to stop
- Name the condition for tool use — "use X tool only when Y"

**Clarifying questions:**

- Tell the agent which missing inputs require a question before proceeding
- Specify what to ask for, not just "ask if unclear"

**Refusal and escalation:**

- Define scope explicitly and provide a refusal message for out-of-scope requests
- Always pair a refusal with an escalation path — don't leave the user stuck

**Instructions become critical in Lessons 14–18** — in multi-agent systems, they are the difference between a pipeline that routes correctly and one that silently misbehaves

---

## 📍 Next Step

06: Pydantic Basics — Learn what Pydantic is, how to define typed models, and why the course uses it for structured agent output.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-05-writing-agent-instructions)

---